# TP1 — Engenharia de Dados com pandas

**CloudDesk — Base de Tickets de Suporte**<br>
**Aluno: Guilherme Reis de Jesus Santos**<br>
**Curso: Engenharia de Software**

## Preparação dos dados

Executar apenas uma vez.

In [1]:
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta

rng = np.random.default_rng(42)
n = 4000
start_date = datetime(2026, 1, 5)

ticket_ids = [f"TCK-{i:05d}" for i in range(1, n + 1)]
cliente_ids = [f"CLI-{i:03d}" for i in rng.integers(1, 181, n)]
dias_offset = np.sort(rng.integers(0, 180, n))
datas = [(start_date + timedelta(days=int(d))).strftime("%Y-%m-%d") for d in dias_offset]
canais = rng.choice(["email", "chat", "telefone"], size=n, p=[0.45, 0.35, 0.20])
categorias = rng.choice(["bug", "duvida_de_uso", "cobranca", "elogio"], size=n, p=[0.35, 0.30, 0.25, 0.10])
prioridades = rng.choice(["baixa", "media", "alta", "critica"], size=n, p=[0.30, 0.35, 0.25, 0.10])
tempo_resposta = np.clip(rng.gamma(shape=2.0, scale=2.2, size=n), 0.2, 48)
tempo_resposta[canais == "chat"] *= 0.6
tempo_resolucao = np.clip(1.8 * tempo_resposta + 3 + rng.normal(0, 1.5, n), 0.5, 120)
satisfacao = 5 - (tempo_resposta / 6) + rng.normal(0, 0.6, n)
satisfacao[canais == "chat"] -= 0.6
satisfacao = np.clip(np.round(satisfacao), 1, 5)

df = pd.DataFrame({
    "ticket_id": ticket_ids,
    "cliente_id": cliente_ids,
    "data_abertura": datas,
    "canal": canais,
    "categoria": categorias,
    "prioridade": prioridades,
    "tempo_resposta_horas": tempo_resposta,
    "tempo_resolucao_horas": tempo_resolucao,
    "satisfacao_cliente": satisfacao,
})

df.loc[rng.choice(n, size=int(n * 0.08), replace=False), "tempo_resposta_horas"] = np.nan
df.loc[rng.choice(n, size=int(n * 0.10), replace=False), "satisfacao_cliente"] = np.nan

def fmt_virgula(v):
    return "" if pd.isna(v) else f"{v:.2f}".replace(".", ",")

df["tempo_resposta_horas"] = df["tempo_resposta_horas"].apply(fmt_virgula)

duplicatas = df.sample(frac=0.02, random_state=42)
df_final = pd.concat([df, duplicatas], ignore_index=True)
df_final.to_csv("tickets_suporte.csv", index=False)

clientes_unicos = sorted(set(cliente_ids))
planos = pd.DataFrame({
    "cliente_id": clientes_unicos,
    "valor_plano": rng.choice([99.0, 199.0, 349.0, 599.0], size=len(clientes_unicos), p=[0.4, 0.3, 0.2, 0.1]),
    "segmento": rng.choice(["starter", "growth", "enterprise"], size=len(clientes_unicos), p=[0.5, 0.35, 0.15]),
})
planos.to_excel("planos_clientes.xlsx", index=False)

chat_ids = df.loc[df["canal"] == "chat", "ticket_id"].tolist()
triagem = [
    {
        "ticket_id": tid,
        "intencao_triagem": str(rng.choice(["duvida", "reclamacao", "solicitacao", "elogio"])),
        "confianca_bot": round(float(rng.uniform(0.55, 0.98)), 2),
    }
    for tid in chat_ids
]

with open("chatbot_triagem.json", "w", encoding="utf-8") as f:
    json.dump(triagem, f, ensure_ascii=False, indent=2)

print("Arquivos gerados: tickets_suporte.csv, planos_clientes.xlsx, chatbot_triagem.json")

Arquivos gerados: tickets_suporte.csv, planos_clientes.xlsx, chatbot_triagem.json


---

## Exercício 1 — Inspeção inicial do CSV

In [2]:
import pandas as pd

df_tickets = pd.read_csv("tickets_suporte.csv")

print("=== shape ===")
print(df_tickets.shape)

print("\n=== dtypes ===")
print(df_tickets.dtypes)

print("\n=== info() ===")
df_tickets.info()

=== shape ===
(4080, 9)

=== dtypes ===
ticket_id                    str
cliente_id                   str
data_abertura                str
canal                        str
categoria                    str
prioridade                   str
tempo_resposta_horas         str
tempo_resolucao_horas    float64
satisfacao_cliente       float64
dtype: object

=== info() ===
<class 'pandas.DataFrame'>
RangeIndex: 4080 entries, 0 to 4079
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ticket_id              4080 non-null   str    
 1   cliente_id             4080 non-null   str    
 2   data_abertura          4080 non-null   str    
 3   canal                  4080 non-null   str    
 4   categoria              4080 non-null   str    
 5   prioridade             4080 non-null   str    
 6   tempo_resposta_horas   3751 non-null   str    
 7   tempo_resolucao_horas  4080 non-null   float64
 8   satisfacao_cl

### Observações:

1. **`tempo_resposta_horas` foi inferida como `object`** — o separador decimal é vírgula (padrão BR), não ponto, então o pandas não conseguiu converter para float automaticamente. Precisa de tratamento antes de qualquer cálculo.

2. **`data_abertura` foi inferida como `object`** — datas armazenadas como texto (string no formato `YYYY-MM-DD`). É necessário converter para `datetime64` para permitir operações temporais como filtragem por período e ordenação cronológica.

3. **`prioridade` e `canal` foram inferidas como `object`** — essas colunas possuem um conjunto fixo e pequeno de valores repetidos (variáveis categóricas). Mantê-las como `object` gasta mais memória do que o tipo `category`, que armazena apenas os valores únicos e usa índices inteiros internamente.

---

## Exercício 2 — Importação das três fontes

In [5]:
import pandas as pd

df_tickets = pd.read_csv("tickets_suporte.csv")
df_planos = pd.read_excel("planos_clientes.xlsx")
df_chatbot = pd.read_json("chatbot_triagem.json")

for nome, df in [("tickets_suporte.csv", df_tickets),
                 ("planos_clientes.xlsx", df_planos),
                 ("chatbot_triagem.json", df_chatbot)]:
    print(f"\n=== {nome} — colunas: {df.shape[1]} ===")
    print(df.head())


=== tickets_suporte.csv — colunas: 9 ===
   ticket_id cliente_id data_abertura     canal      categoria prioridade  \
0  TCK-00001    CLI-017    2026-01-05      chat  duvida_de_uso      media   
1  TCK-00002    CLI-140    2026-01-05     email       cobranca      media   
2  TCK-00003    CLI-118    2026-01-05  telefone            bug      baixa   
3  TCK-00004    CLI-079    2026-01-05     email            bug    critica   
4  TCK-00005    CLI-078    2026-01-05      chat       cobranca      baixa   

  tempo_resposta_horas  tempo_resolucao_horas  satisfacao_cliente  
0                 1,54               6.833510                 4.0  
1                 2,05               6.664003                 5.0  
2                 2,04               6.184528                 4.0  
3                 5,19              10.471215                 3.0  
4                 0,63               3.049426                 4.0  

=== planos_clientes.xlsx — colunas: 3 ===
  cliente_id  valor_plano segmento
0    CLI-

---

## Exercício 3 — Conversão de tipos

In [ ]:

print("Tipos antes da conversão:")
print(df_tickets[["data_abertura", "prioridade", "tempo_resposta_horas"]].dtypes)

df_tickets["data_abertura"] = pd.to_datetime(df_tickets["data_abertura"])

df_tickets["tempo_resposta_horas"] = (
    df_tickets["tempo_resposta_horas"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .replace("", float("nan"))  
    .astype(float)
)

df_tickets["prioridade"] = df_tickets["prioridade"].astype("category")

print("\nTipos depois da conversão:")
print(df_tickets[["data_abertura", "prioridade", "tempo_resposta_horas"]].dtypes)

Tipos antes da conversão:
data_abertura           datetime64[ns]
prioridade                    category
tempo_resposta_horas           float64
dtype: object

Tipos depois da conversão:
data_abertura           datetime64[ns]
prioridade                    category
tempo_resposta_horas           float64
dtype: object


### Por que `object` seria uma decisão ruim para `prioridade`?

Com `object`, o pandas armazena uma string Python completa para cada uma das ~4 000+ linhas, mesmo que existam apenas 4 valores únicos (`"baixa"`, `"media"`, `"alta"`, `"crítica"`). Convertendo para `category`, o pandas guarda os 4 strings apenas uma vez e usa um array de inteiros compacto para referenciar cada linha economizando memória proporcional ao número de repetições e acelerando agrupamentos. Em datasets com centenas de milhares de linhas o ganho de memória pode chegar a 10× ou mais.

---

## Exercício 4 — Seleção com `loc` e `iloc`

In [8]:

df_subset = df_tickets.loc[:, ["ticket_id", "categoria", "tempo_resposta_horas"]]

df_semana2 = df_subset.iloc[500:801]

print(f"Shape do subconjunto (semana 2, 3 colunas): {df_semana2.shape}")
print(df_semana2.head())

Shape do subconjunto (semana 2, 3 colunas): (301, 3)
     ticket_id      categoria  tempo_resposta_horas
500  TCK-00501            bug                   NaN
501  TCK-00502       cobranca                  4.20
502  TCK-00503            bug                  1.39
503  TCK-00504  duvida_de_uso                  2.13
504  TCK-00505       cobranca                   NaN


### `loc` vs `iloc` num pipeline real

Usa-se **`loc`** quando se referencia colunas ou índices pelos seus **rótulos** (nomes de colunas, datas, IDs) — o que torna o código legível e robusto a reordenações do DataFrame; usa-se **`iloc`** quando se quer selecionar por **posição numérica** (ex.: as primeiras N linhas ou um slice fixo por índice inteiro), independente dos rótulos.

---

## Exercício 5 — Filtros booleanos compostos

In [ ]:

mask = (df_tickets["prioridade"] == "alta") & (df_tickets["canal"] == "email")
df_alta_email_bool = df_tickets[mask]

df_alta_email_query = df_tickets.query('prioridade == "alta" and canal == "email"')

print(f"Booleano: {len(df_alta_email_bool)} linhas")
print(f"query():  {len(df_alta_email_query)} linhas")
print(f"Resultado igual: {len(df_alta_email_bool) == len(df_alta_email_query)}")

Booleano: 494 linhas
query():  494 linhas
Resultado igual: True


### Vantagem de `query()` sobre a notação booleana

`query()` permite escrever a condição como uma string de texto simples, sem precisar repetir o nome do DataFrame ou usar parênteses ao redor de cada condição — o que melhora muito a legibilidade quando há várias condições encadeadas.

---

## Exercício 6 — Tratamento de valores ausentes

In [ ]:

print("Valores ausentes por coluna:")
print(df_tickets.isnull().sum())

df_tickets["tempo_resposta_horas"] = df_tickets.groupby("categoria")["tempo_resposta_horas"].transform(
    lambda x: x.fillna(x.median())
)

linhas_antes = len(df_tickets)
df_tickets = df_tickets.dropna(subset=["ticket_id"])
linhas_depois = len(df_tickets)

print(f"\nLinhas removidas por ticket_id nulo: {linhas_antes - linhas_depois}")
print("\nNulos restantes após tratamentos:")
print(df_tickets.isnull().sum())

Valores ausentes por coluna:
ticket_id                  0
cliente_id                 0
data_abertura              0
canal                      0
categoria                  0
prioridade                 0
tempo_resposta_horas     329
tempo_resolucao_horas      0
satisfacao_cliente       407
dtype: int64

Linhas removidas por ticket_id nulo: 0

Nulos restantes após tratamentos:
ticket_id                  0
cliente_id                 0
data_abertura              0
canal                      0
categoria                  0
prioridade                 0
tempo_resposta_horas       0
tempo_resolucao_horas      0
satisfacao_cliente       407
dtype: int64


---

## Exercício 7 — Remoção de duplicatas

In [ ]:
total_original = len(df_tickets)

n_duplicatas = df_tickets.duplicated(subset=["ticket_id"]).sum()
print(f"Linhas duplicadas (mesmo ticket_id): {n_duplicatas}")

df_tickets = df_tickets.drop_duplicates(subset=["ticket_id"], keep="first")

removidas = total_original - len(df_tickets)
pct = removidas / total_original * 100
print(f"Linhas após remoção: {len(df_tickets)}")
print(f"Removidas: {removidas} ({pct:.2f}% do dataset original)")

Linhas duplicadas (mesmo ticket_id): 80
Linhas após remoção: 4000
Removidas: 80 (1.96% do dataset original)


### Relatório de remoção de duplicatas

O dataset original (`tickets_suporte.csv`) foi gerado com 2% de duplicatas de reenvio concatenadas ao final do arquivo. Após aplicar `drop_duplicates(subset=['ticket_id'], keep='first')`, **~80 linhas foram removidas**, representando aproximadamente **2% do total de registros**. Isso é esperado e alinhado com a especificação de geração dos dados. Manter as duplicatas inflaria contagens de tickets e médias de tempo de resposta, distorcendo qualquer relatório gerado a partir desse dataset.

---

## Exercício 8 — Consolidação das três fontes

In [3]:

import pandas as pd

df_tickets = pd.read_csv("tickets_suporte.csv")
df_planos = pd.read_excel("planos_clientes.xlsx")
df_chatbot = pd.read_json("chatbot_triagem.json")

df_tickets["tempo_resposta_horas"] = (
    df_tickets["tempo_resposta_horas"]
    .astype(str).str.replace(",", ".", regex=False)
    .replace("", float("nan")).astype(float)
)
df_tickets = df_tickets.drop_duplicates(subset=["ticket_id"], keep="first")

df_merged = pd.merge(df_tickets, df_planos, on="cliente_id", how="left")
print(f"Após merge tickets + planos: {df_merged.shape}")

df_consolidado = pd.merge(df_merged, df_chatbot, on="ticket_id", how="left")
print(f"DataFrame consolidado final: {df_consolidado.shape}")
print(df_consolidado.columns.tolist())

print(f"Shape chatbot: {df_chatbot.shape}")
print(f"Shape planos:  {df_planos.shape}")
print(f"Shape final:   {df_consolidado.shape}")

Após merge tickets + planos: (4000, 11)
DataFrame consolidado final: (4000, 13)
['ticket_id', 'cliente_id', 'data_abertura', 'canal', 'categoria', 'prioridade', 'tempo_resposta_horas', 'tempo_resolucao_horas', 'satisfacao_cliente', 'valor_plano', 'segmento', 'intencao_triagem', 'confianca_bot']
Shape chatbot: (1397, 3)
Shape planos:  (180, 3)
Shape final:   (4000, 13)


---

## Exercício 9 — Agregação por categoria

In [10]:
resumo = (
    df_consolidado
    .groupby("categoria", observed=True)
    .agg(
        media_tempo_resposta=("tempo_resposta_horas", "mean"),
        total_tickets=("ticket_id", "count"),
    )
    .sort_values("media_tempo_resposta", ascending=False)
    .round(2)
)

print(resumo)

categoria_prioritaria = resumo.index[0]
print(f"\nCategoria com maior tempo médio de resposta: {categoria_prioritaria}")

               media_tempo_resposta  total_tickets
categoria                                         
elogio                         4.03            393
cobranca                       3.93           1028
bug                            3.74           1387
duvida_de_uso                  3.72           1192

Categoria com maior tempo médio de resposta: elogio
